In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import math
import networkx as nx
import copy
import simpy
import matplotlib.animation as animation
from matplotlib.patches import Circle
import pandas as pd
import os
import datetime
import csv

# ENERGY MODEL 
class EnergyModel:
    """Same energy model used as used in the NESTED"""
    
    # Physical constants
    GRAVITY = 9.81
    AIR_DENSITY = 1.225
    UAV_MASS = 3.49
    NUM_PROPELLERS = 6
    PROPELLER_RADIUS = 0.21
    LIFT_COEFF = 0.0435
    DRAG_COEFF = 0.002
    PITCH_COEFF = 0.00016
    PROPELLER_PITCH = 0.5
    
    # Communication parameters
    DATA_PACKET_SIZE = 100
    CONTROL_PACKET_SIZE = 5
    DATA_RATE = 250_000
    RADIO_TX_POWER = 62.64 / 1000
    RADIO_RX_POWER = 57.4 / 1000
    RADIO_IDLE_POWER = 62.64 / 1000
    RADIO_SLEEP_POWER = 72e-6
    PROCESSING_POWER = 2.0
    PACKET_PROCESSING_OVERHEAD = 0.001
    TIME_SLOT = 0.5
    MAX_TX_POWER = 0.1
    MAX_MOTOR_POWER = 5000.0
    MAX_ACCELERATION = 5.0
    
    @staticmethod
    def hovering_power():
        m = EnergyModel.UAV_MASS
        g = EnergyModel.GRAVITY
        rho = EnergyModel.AIR_DENSITY
        xi = EnergyModel.NUM_PROPELLERS
        r = EnergyModel.PROPELLER_RADIUS
        CL = EnergyModel.LIFT_COEFF
        CD = EnergyModel.DRAG_COEFF
        induced = np.sqrt((2 * m * g) / (xi * rho * np.pi * r ** 2 * CL))
        return (m * g * CD / CL) * induced
    
    @staticmethod
    def limit_acceleration(v, dv):
        max_dv = EnergyModel.MAX_ACCELERATION * EnergyModel.TIME_SLOT
        norm = np.linalg.norm(dv)
        if norm > max_dv:
            dv = dv / norm * max_dv
        return dv
    
    @staticmethod
    def kinetic_power(velocity, delta_velocity):
        delta_velocity = EnergyModel.limit_acceleration(velocity, delta_velocity)
        m = EnergyModel.UAV_MASS
        CM = EnergyModel.PITCH_COEFF
        J = EnergyModel.PROPELLER_PITCH
        CL = EnergyModel.LIFT_COEFF
        delta = EnergyModel.TIME_SLOT
        
        v_norm = np.linalg.norm(velocity)
        dv_norm = np.linalg.norm(delta_velocity)
        
        coeff = (2 * np.pi * m * CM) / (delta * J * CL)
        return coeff * v_norm * dv_norm
    
    @staticmethod
    def flying_power(velocity, delta_velocity):
        p_hover = EnergyModel.hovering_power()
        p_kinetic = EnergyModel.kinetic_power(velocity, delta_velocity)
        power = p_hover + p_kinetic
        return min(power, EnergyModel.MAX_MOTOR_POWER)
    
    @staticmethod
    def processing_power():
        return EnergyModel.PROCESSING_POWER
    
    @staticmethod
    def calculate_packet_transmission_time(packet_size_bytes):
        bits = packet_size_bytes * 8
        return bits / EnergyModel.DATA_RATE
    
    @staticmethod
    def calculate_message_energy_cost(message_type='control'):
        if message_type == 'data':
            packet_size = EnergyModel.DATA_PACKET_SIZE
        else:
            packet_size = EnergyModel.CONTROL_PACKET_SIZE
        
        tx_time = EnergyModel.calculate_packet_transmission_time(packet_size)
        processing_time = EnergyModel.PACKET_PROCESSING_OVERHEAD
        
        tx_energy = EnergyModel.RADIO_TX_POWER * tx_time
        processing_energy = EnergyModel.PROCESSING_POWER * processing_time
        
        return tx_energy + processing_energy
    
    @staticmethod
    def total_energy(velocity, delta_velocity, tx_power, message_type='data'):
        tx_power = min(tx_power, EnergyModel.MAX_TX_POWER)
        p_f = EnergyModel.flying_power(velocity, delta_velocity)
        p_processing = EnergyModel.processing_power()
        total_power = p_f + p_processing + tx_power
        energy = EnergyModel.TIME_SLOT * total_power
        return max(0.0, energy)


class Drone:
    def __init__(self, env, drone_id, position, energy, uav_height, network):
        self.env = env
        self.drone_id = drone_id
        self.position = position
        self.energy = energy
        self.max_energy = energy
        self.uav_height = uav_height
        self.network = network
        self.neighbors = set()
        self.failed = False
        self.velocity = np.array([0.0, 0.0, 0.0])
        self.previous_position = position
        
        # Game theory parameters
        self.learning_rate = 0.001
        self.step_size = 0.0009
        self.exploration_rate = 0.3
        self.utility = 0
        self.action_history = []
        self.utility_history = []
        self.actions_taken = 0

    def update_velocity(self, new_position):
        old_pos = np.array(self.previous_position)
        new_pos = np.array(new_position)
        displacement = new_pos - old_pos
        velocity = displacement / EnergyModel.TIME_SLOT if EnergyModel.TIME_SLOT > 0 else np.array([0.0, 0.0, 0.0])
        self.velocity = velocity
        self.previous_position = new_position
        return velocity
    
    def consume_energy(self, energy_amount):
        self.energy = max(0, self.energy - energy_amount)
        if self.energy <= 0:
            self.fail()
            return True
        return False
    
    def fail(self):
        self.failed = True
        self.energy = 0
        self.neighbors = set()
        print(f"    [Death] Drone {self.drone_id} has DIED (energy depletion)")


class User:
    def __init__(self, env, user_id, position, network):
        self.env = env
        self.user_id = user_id
        self.position = position
        self.network = network
        self.connected_drone = None
        self.movement_data = None


class DroneNetwork:
    def __init__(self, env, run_id):
        self.env = env
        self.run_id = run_id
        self.field_size = 30000
        self.rows = 6
        self.cols = 10
        self.no_drones = 12
        self.no_people = 200
        self.uav_height = 1500
        self.grid_width = self.field_size // self.cols
        self.grid_height = self.field_size // self.rows
        self.access_link_range = 3300
        self.backhaul_link_range = 8200
        self.a = 4.88
        self.b = 0.429
        self.eta_L = 0.1
        self.eta_N = 21
        self.fc = 2e9
        self.c = 3e8
        self.tx_power = 0.5
        self.access_snr_threshold = 17
        self.backhaul_snr_threshold = 10
        self.noise_power = 10 ** (-170 / 10)
        self.drones = []
        self.users = []
        self.drone_positions = []
        self.people_positions = []
        self.neighbors = {i: set() for i in range(self.no_drones)}
        self.drone_energy = [125000.0] * self.no_drones  # Full battery
        self.backhaul_links = []
        self.access_links = []
        self.triangle_count = 0
        self.covered_users = []
        self.links = []
        self.max_sim_time = 45
        
        # Energy parameters
        self.max_battery_energy = 125000.0
        self.energy_threshold_death = 100
        
        self.metrics_over_time = {
            'time': [], 'coverage': [], 'components': [],
            'num_failed': [], 'efficiency': [], 'avg_energy': [], 'min_energy': []
        }

        # Game theory parameters
        self.temperature = 1.0
        self.learning_rate_initial = 0.001
        self.step_size = 0.0009
        self.exploration_rate_max = 0.8
        self.exploration_rate_min = 0.1
        self.epsilon = 0.1
        self.quantization_orders = {'L': 8, 'K': 4}
        self.action_space_sizes = []
        self.velocity = 100
        self.time_lap_duration = 2.0
        self.current_time_lap = 0
        self.total_time_laps = int(self.max_sim_time / self.time_lap_duration)


        self.last_link_hash = None
        
        # Energy logging
        self.energy_log = []
        
        self.initialize_game_parameters()

    def load_users_from_csv(self, filename):
        """Load users from CSV file"""
        print(f"[{self.env.now}] [Run {self.run_id}] Loading users from CSV: {filename}")
        try:
            df = pd.read_csv(filename)
            print(f"Successfully loaded CSV with {len(df)} rows")
            
            user_ids = df['uid'].unique()
            print(f"Found {len(user_ids)} unique users")
            
            self.no_people = min(self.no_people, len(user_ids))
            
            users = []
            user_data_map = {}
            
            for i, user_id in enumerate(user_ids[:self.no_people]):
                user_df = df[df['uid'] == user_id].sort_values('step')
                initial_data = user_df[user_df['step'] == 0]
                
                if len(initial_data) > 0:
                    initial_row = initial_data.iloc[0]
                    position = (float(initial_row['x']), float(initial_row['y']))
                    user = User(self.env, i, position, self)
                    users.append(user)
                    
                    user_data_map[i] = user_df
            
            print(f"Initialized {len(users)} users from CSV")
            return users, user_data_map
            
        except Exception as e:
            print(f"Error loading CSV: {e}")
            return None, None

    def update_user_positions(self, time_step, user_data_map):
        """Update user positions based on CSV data"""
        for user in self.users:
            if user.user_id in user_data_map:
                user_df = user_data_map[user.user_id]
                # Find closest time step
                time_diff = np.abs(user_df['sim_time_min'] - time_step)
                closest_idx = time_diff.idxmin()
                if time_diff[closest_idx] <= 1.0:
                    new_pos = (user_df.loc[closest_idx, 'x'], user_df.loc[closest_idx, 'y'])
                    user.position = new_pos
        
        self.people_positions = [user.position for user in self.users]

    def initialize_action_spaces(self):
        L = self.quantization_orders['L']
        K = self.quantization_orders['K']
        action_space_size = 1 + L * (2 * K + 1)
        self.action_space_sizes = [action_space_size for _ in range(self.no_drones)]
        print(f"[{self.env.now}] [Run {self.run_id}] Action space size: {action_space_size}")

    def get_connectivity_probability(self, drone_i, drone_j, action_i, action_j):
        pos_i = self.calculate_new_position(self.drones[drone_i].position, action_i)
        pos_j = self.calculate_new_position(self.drones[drone_j].position, action_j)
        distance = self.euclidean_distance(pos_i, pos_j)
        
        if distance > self.backhaul_link_range:
            return 0.0
        
        pl = self.calculate_path_loss_backhaul(pos_i, pos_j)
        snr = self.calculate_snr(pl)
        eta_ij = max(0, self.backhaul_snr_threshold - snr)
        connectivity_prob = 1 / (1 + self.epsilon ** eta_ij)
        return connectivity_prob

    def generate_action_space(self, drone_id):
        L = self.quantization_orders['L']
        K = self.quantization_orders['K']
        action_space = []
        action_space.append((0, 0, 0))
        
        for l in range(L):
            for k in range(-K, K + 1):
                omega_angle = (2 * np.pi / L) * l
                phi_angle = (np.pi / (2 * K)) * k
                action_space.append((self.velocity, omega_angle, phi_angle))
        
        return action_space

    def calculate_new_position(self, current_position, action):
        x, y, z = current_position
        v, omega_angle, phi_angle = action
        dt = self.time_lap_duration * 60
        
        if v > 0:
            new_x = x + v * dt * np.cos(phi_angle) * np.sin(omega_angle)
            new_y = y + v * dt * np.cos(phi_angle) * np.cos(omega_angle)
            new_z = z + v * dt * np.sin(phi_angle)
            new_x = max(0, min(self.field_size, new_x))
            new_y = max(0, min(self.field_size, new_y))
            new_z = max(500, min(2500, new_z))
        else:
            new_x, new_y, new_z = x, y, z
        
        return (new_x, new_y, new_z)

    def get_restricted_action_space(self, drone_id, current_action):
        full_action_space = self.generate_action_space(drone_id)
        restricted_actions = []
        current_position = self.drones[drone_id].position
        
        for action in full_action_space:
            new_position = self.calculate_new_position(current_position, action)
            
            if new_position[2] < 500 or new_position[2] > 2500:
                continue
            
            if (new_position[0] < 0 or new_position[0] > self.field_size or
                new_position[1] < 0 or new_position[1] > self.field_size):
                continue
            
            communication_graph = self.get_communication_graph()
            if drone_id in communication_graph:
                current_neighbors = list(communication_graph.neighbors(drone_id))
                if current_neighbors:
                    can_communicate = False
                    for neighbor_id in current_neighbors:
                        if self.drone_energy[neighbor_id] == 0:
                            continue
                        neighbor_pos = self.drones[neighbor_id].position
                        distance = self.euclidean_distance(new_position, neighbor_pos)
                        if distance <= self.backhaul_link_range:
                            can_communicate = True
                            break
                    
                    if not can_communicate and action != (0, 0, 0):
                        continue
            
            restricted_actions.append(action)
        
        if (0, 0, 0) not in restricted_actions:
            restricted_actions.append((0, 0, 0))
        
        return restricted_actions

    def calculate_individual_coverage_area(self, drone_id, position):
        z = position[2]
        radius = z * np.tan(np.radians(30))
        return np.pi * radius**2

    def calculate_cooperative_coverage_area(self, drone_positions, drone_ids):
        if not drone_ids:
            return 0.0
        
        sample_area_size = self.field_size + 5000
        num_samples = 10000
        covered_points = 0
        
        for _ in range(num_samples):
            x = random.uniform(-2500, sample_area_size + 2500)
            y = random.uniform(-2500, sample_area_size + 2500)
            
            is_covered = False
            for drone_id in drone_ids:
                if self.drone_energy[drone_id] == 0:
                    continue
                drone_pos = drone_positions[drone_id]
                horizontal_distance = np.sqrt((x - drone_pos[0])**2 + (y - drone_pos[1])**2)
                coverage_radius = drone_pos[2] * np.tan(np.radians(30))
                
                if horizontal_distance <= coverage_radius:
                    is_covered = True
                    break
            
            if is_covered:
                covered_points += 1
        
        total_sample_area = sample_area_size**2
        coverage_area = (covered_points / num_samples) * total_sample_area
        return coverage_area

    def calculate_utility(self, drone_id, action, neighbor_actions):
        communication_graph = self.get_communication_graph()
        new_position = self.calculate_new_position(self.drones[drone_id].position, action)
        
        temp_positions = {}
        for i in range(self.no_drones):
            temp_positions[i] = self.drones[i].position
        temp_positions[drone_id] = new_position
        
        for neighbor_id, neighbor_action in neighbor_actions.items():
            if neighbor_id != drone_id and self.drone_energy[neighbor_id] > 0:
                temp_positions[neighbor_id] = self.calculate_new_position(
                    self.drones[neighbor_id].position, neighbor_action)
        
        cooperative_drones = {drone_id}
        if drone_id in communication_graph:
            cooperative_drones.update(communication_graph.neighbors(drone_id))
        
        cooperative_drones = {d for d in cooperative_drones if self.drone_energy[d] > 0}
        utility = self.calculate_cooperative_coverage_area(temp_positions, cooperative_drones)
        return utility

    def consume_movement_energy(self, drone, old_position, new_position):
        """Consume energy for movement"""
        if drone.failed:
            return 0
        
        velocity = drone.update_velocity(new_position)
        delta_velocity = velocity
        tx_power = self.tx_power
        
        energy_consumed = EnergyModel.total_energy(velocity, delta_velocity, tx_power, 'data')
        died = drone.consume_energy(energy_consumed)
        
        if died:
            self.drone_energy[drone.drone_id] = 0
        
        return energy_consumed

    def consume_hovering_energy(self, drone):
        """Consume energy for hovering"""
        if drone.failed:
            return 0
        
        velocity = np.array([0.0, 0.0, 0.0])
        delta_velocity = np.array([0.0, 0.0, 0.0])
        tx_power = self.tx_power
        
        energy_consumed = EnergyModel.total_energy(velocity, delta_velocity, tx_power, 'control')
        died = drone.consume_energy(energy_consumed)
        
        if died:
            self.drone_energy[drone.drone_id] = 0
        
        return energy_consumed

    def run_time_lap(self, time_lap):
        print(f"[{self.env.now}] [Run {self.run_id}] Starting time lap {time_lap+1}/{self.total_time_laps}")
        
        active_drones = [i for i in range(self.no_drones) if self.drone_energy[i] > 0]
        if not active_drones:
            return 0
        
        drone_id = random.choice(active_drones)
        print(f"[{self.env.now}] Selected drone {drone_id} for this time lap")
        
        communication_graph = self.get_communication_graph()
        exploration_rate = self.update_exploration_rate(drone_id, time_lap)
        
        current_neighbors = []
        if drone_id in communication_graph:
            current_neighbors = list(communication_graph.neighbors(drone_id))
        
        current_action = self.drones[drone_id].action_history[-1] if self.drones[drone_id].action_history else (0, 0, 0)
        restricted_actions = self.get_restricted_action_space(drone_id, current_action)
        
        if not restricted_actions:
            self.drones[drone_id].action_history.append(current_action)
            # Consume hovering energy
            self.consume_hovering_energy(self.drones[drone_id])
            return 0
        
        if random.random() < exploration_rate:
            available_actions = [a for a in restricted_actions if a != current_action]
            if available_actions:
                trial_action = random.choice(available_actions)
            else:
                trial_action = current_action
            selected_action_candidate = trial_action
            print(f"[{self.env.now}] Drone {drone_id}: EXPLORING - selected trial action {trial_action}")
        else:
            selected_action_candidate = current_action
            print(f"[{self.env.now}] Drone {drone_id}: EXPLOITING - keeping current action {current_action}")
        
        neighbor_actions = {}
        for neighbor_id in current_neighbors:
            if self.drones[neighbor_id].action_history:
                neighbor_actions[neighbor_id] = self.drones[neighbor_id].action_history[-1]
            else:
                neighbor_actions[neighbor_id] = (0, 0, 0)
        
        if selected_action_candidate != current_action:
            current_utility = self.calculate_utility(drone_id, current_action, neighbor_actions)
            trial_utility = self.calculate_utility(drone_id, selected_action_candidate, neighbor_actions)
            
            selected_action, selected_utility = self.select_action_with_strategy(
                drone_id, current_action, selected_action_candidate, current_utility, trial_utility
            )
            
            print(f"[{self.env.now}] Drone {drone_id}: Binary log-linear learning - "
                  f"Current utility: {current_utility:.2f}, Trial utility: {trial_utility:.2f}")
        else:
            selected_action = current_action
            selected_utility = self.calculate_utility(drone_id, current_action, neighbor_actions)
        
        self.drones[drone_id].action_history.append(selected_action)
        self.drones[drone_id].utility_history.append(selected_utility)
        
        old_position = self.drones[drone_id].position
        new_position = self.calculate_new_position(self.drones[drone_id].position, selected_action)
        
        # Consume energy for movement
        if selected_action != (0, 0, 0):
            energy_consumed = self.consume_movement_energy(self.drones[drone_id], old_position, new_position)
            print(f"[{self.env.now}] Drone {drone_id} movement energy: {energy_consumed:.2f}J, remaining: {self.drones[drone_id].energy:.0f}J")
        else:
            energy_consumed = self.consume_hovering_energy(self.drones[drone_id])
            print(f"[{self.env.now}] Drone {drone_id} hovering energy: {energy_consumed:.2f}J, remaining: {self.drones[drone_id].energy:.0f}J")
        
        self.drones[drone_id].position = new_position
        self.drone_positions[drone_id] = new_position
        
        action_taken = 1 if selected_action != current_action else 0
        print(f"[{self.env.now}] Drone {drone_id} final action: {selected_action}, utility: {selected_utility:.2f}")
        
        return action_taken

    def initialize_game_parameters(self):
        self.learning_rate_initial = 0.001
        self.step_size = 0.0009
        self.exploration_rate_max = 0.8
        self.exploration_rate_min = 0.01
        self.epsilon = 0.1
        self.initialize_action_spaces()
        
        print(f"[{self.env.now}] [Run {self.run_id}] Game parameters initialized")
        print(f"  Learning rate: {self.learning_rate_initial}, Step size: {self.step_size}")
        print(f"  Exploration rate range: [{self.exploration_rate_min}, {self.exploration_rate_max}]")

    def initialize_drones_for_game(self):
        for drone in self.drones:
            drone.action_history = [(0, 0, 0)]
            drone.utility_history = [0.0]
            drone.exploration_rate = self.exploration_rate_max
            drone.actions_taken = 0
        
        print(f"[{self.env.now}] [Run {self.run_id}] Drones initialized for game-theoretic learning")

    def get_communication_graph(self):
        G = nx.Graph()
        active_drones = [i for i in range(self.no_drones) if self.drone_energy[i] > 0]
        G.add_nodes_from(active_drones)
        
        for i, j in self.backhaul_links:
            if self.drone_energy[i] > 0 and self.drone_energy[j] > 0:
                failure_prob = 0.05
                if random.random() > failure_prob:
                    G.add_edge(i, j)
        
        return G

    def euclidean_distance(self, p1, p2):
        if len(p1) == 2 and len(p2) == 2:
            return np.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)
        return np.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2 + (p1[2]-p2[2])**2)

    def calculate_path_loss_access(self, user_pos, drone_pos):
        x_u, y_u = user_pos
        x_d, y_d, h_d = drone_pos
        delta = math.sqrt((x_d - x_u)**2 + (y_d - y_u)**2)
        if delta < 1.0:
            delta = 1.0
        theta = math.degrees(math.atan(h_d / delta))
        distance = math.sqrt(delta**2 + h_d**2)
        phi_L = 20 * math.log10(4 * math.pi * self.fc * distance / self.c) + self.eta_L
        phi_N = 20 * math.log10(4 * math.pi * self.fc * distance / self.c) + self.eta_N
        p_L = 1 / (1 + self.a * math.exp(-self.b * (theta - self.a)))
        path_loss = p_L * phi_L + (1 - p_L) * phi_N
        return path_loss

    def calculate_path_loss_backhaul(self, drone1, drone2):
        distance = self.euclidean_distance(drone1, drone2)
        if distance < 1.0:
            return float('inf')
        path_loss = 20 * math.log10(4 * math.pi * self.fc * distance / self.c) + self.eta_L
        return path_loss

    def calculate_snr(self, path_loss_db):
        if path_loss_db == float('inf'):
            return -float('inf')
        path_loss_linear = 10 ** (path_loss_db / 10)
        received_power = self.tx_power / path_loss_linear
        if received_power <= 0:
            return -float('inf')
        snr = received_power / self.noise_power
        if snr <= 0:
            return -float('inf')
        return 10 * math.log10(snr)

    def build_access_links(self):
        self.access_links.clear()
        for u_idx, user in enumerate(self.users):
            user.connected_drone = None
            best_snr = -float('inf')
            best_drone = None
            
            for d_idx, drone in enumerate(self.drones):
                if self.drone_energy[d_idx] == 0:
                    continue
                
                distance = self.euclidean_distance(user.position, drone.position[:2])
                if distance > self.access_link_range:
                    continue
                
                pl = self.calculate_path_loss_access(user.position, drone.position)
                snr = self.calculate_snr(pl)
                
                if snr >= self.access_snr_threshold and snr > best_snr:
                    best_snr = snr
                    best_drone = d_idx
            
            if best_drone is not None:
                self.access_links.append((u_idx, best_drone))
                user.connected_drone = best_drone
        
        self.links = self.backhaul_links + self.access_links

    def build_backhaul_links(self):
        self.backhaul_links.clear()
        self.neighbors = {i: set() for i in range(self.no_drones)}
        
        for i in range(self.no_drones):
            if self.drone_energy[i] == 0:
                continue
            for j in range(i + 1, self.no_drones):
                if self.drone_energy[j] == 0:
                    continue
                distance = self.euclidean_distance(self.drone_positions[i], self.drone_positions[j])
                if distance > self.backhaul_link_range:
                    continue
                pl = self.calculate_path_loss_backhaul(self.drone_positions[i], self.drone_positions[j])
                snr = self.calculate_snr(pl)
                if snr >= self.backhaul_snr_threshold:
                    self.backhaul_links.append((i, j))
                    self.neighbors[i].add(j)
                    self.neighbors[j].add(i)
                    self.drones[i].neighbors.add(j)
                    self.drones[j].neighbors.add(i)
        
        self.links = self.backhaul_links + self.access_links
        print(f"Rebuilt {len(self.backhaul_links)} backhaul links")

    def calculate_coverage(self):
        covered_users = set()
        for u_idx, d_idx in self.access_links:
            if self.drone_energy[d_idx] > 0:
                covered_users.add(u_idx)
        
        coverage = len(covered_users) / len(self.users) if self.users else 0
        self.covered_users = list(covered_users)
        return coverage


        


    def count_failed_drones(self):
        return sum(1 for i in range(self.no_drones) if self.drone_energy[i] == 0)

    def calculate_network_efficiency(self):
        V = [i for i in range(self.no_drones) if self.drone_energy[i] > 0]
        N = len(V)
        if N < 2:
            return 0.0
        G = nx.Graph()
        G.add_nodes_from(V)
        G.add_edges_from(self.backhaul_links)
        total = 0.0
        for i_idx in range(N):
            for j_idx in range(i_idx + 1, N):
                i_node = V[i_idx]
                j_node = V[j_idx]
                try:
                    d = nx.shortest_path_length(G, i_node, j_node)
                    total += 2 / d
                except nx.NetworkXNoPath:
                    pass
        denominator = N * (N - 1)
        return total / denominator if denominator > 0 else 0.0

    def run_game_theoretic_simulation(self, user_mobility_file):
        def simulation_process(env):
            print(f"[{env.now}] [Run {self.run_id}] Initializing game-theoretic approach...")
            
            self.initialize_action_spaces()
            
            # Load users from CSV
            print(f"[{env.now}] [Run {self.run_id}] Loading users from CSV...")
            users, user_data_map = self.load_users_from_csv(user_mobility_file)
            
            if users is None:
                print(f"ERROR: Failed to load users from CSV")
                return
            
            self.users = users
            self.people_positions = [user.position for user in self.users]
            
            yield self.env.timeout(1)
            
            # Place drones
            print(f"[{env.now}] [Run {self.run_id}] Placing drones...")
            self.drone_positions = self.place_drones(self.no_drones, self.uav_height)
            self.drones = [Drone(self.env, i, pos, self.drone_energy[i], self.uav_height, self)
                          for i, pos in enumerate(self.drone_positions)]
            
            yield self.env.timeout(1)
            
            # Initialize drone actions
            for drone in self.drones:
                drone.action_history.append((0, 0, 0))
                drone.exploration_rate = self.exploration_rate_max
            
            # Initial links
            self.build_backhaul_links()
            self.build_access_links()
            
            # Main loop
            topology_changed = False
            
            for t in range(self.max_sim_time + 1):
                topology_changed = False
                
                # Update user positions from CSV
                if t > 0:
                    self.update_user_positions(t, user_data_map)
                 
                # Game theory every 2 min    
                if t > 0 and t % 2 == 0:
                    time_lap = (t // 2) - 1
                    actions_taken = self.run_time_lap(time_lap)    
                
                    
                    if actions_taken > 0:
                        self.build_backhaul_links()
                        self.build_access_links()
                        topology_changed = True
                
                # Consume hovering energy for drones that didn't move
                if t % 10 != 0 or t == 0:
                    for drone in self.drones:
                        if not drone.failed:
                            self.consume_hovering_energy(drone)
                
                # Calculate metrics
                coverage = self.calculate_coverage()
                components = nx.number_connected_components(self.get_communication_graph())
                num_failed = self.count_failed_drones()
                efficiency = self.calculate_network_efficiency()
                
                # Energy metrics
                active_drones = [d for d in self.drones if not d.failed]
                avg_energy = np.mean([d.energy for d in active_drones]) if active_drones else 0
                min_energy = min([d.energy for d in active_drones]) if active_drones else 0
                
                
                
                # Store metrics
                self.metrics_over_time['time'].append(t)

                self.metrics_over_time['coverage'].append(coverage)
                self.metrics_over_time['components'].append(components)
                self.metrics_over_time['num_failed'].append(num_failed)
                self.metrics_over_time['efficiency'].append(efficiency)
                self.metrics_over_time['avg_energy'].append(avg_energy)
                self.metrics_over_time['min_energy'].append(min_energy)
                
                
                yield self.env.timeout(1)
            
            # Final output
            print(f"\n[Run {self.run_id}] Simulation complete")
            print(f"  Final Avg Energy: {avg_energy:.0f}J, Min Energy: {min_energy:.0f}J")
            
            self.export_to_csv()
        
        self.env.process(simulation_process(self.env))
        self.env.run(until=self.max_sim_time + 10)
        
        return self.metrics_over_time

    def place_drones(self, no_drones, uav_height):
        drone_positions = []
        for _ in range(no_drones):
            x = random.uniform(0, self.field_size)
            y = random.uniform(0, self.field_size)
            drone_positions.append((x, y, uav_height))
        return drone_positions

    def export_to_csv(self):
        """Export data to individual CSV file for this iteration"""
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        csv_data = []
        metrics = self.metrics_over_time
        times = metrics['time']
        coverages = metrics['coverage']
        components = metrics['components']
        num_faileds = metrics['num_failed']
        efficiencies = metrics['efficiency']
        avg_energies = metrics['avg_energy']
        min_energies = metrics['min_energy']
        
        for i, t in enumerate(times):
            row = {
                'Iteration No': self.run_id,
                'Coverage': coverages[i] * 100,
                'No of connected components': components[i],
                'Simulation time': t,
                'No of drone failed': num_faileds[i],
                'Network efficiency': round(efficiencies[i], 4),
                'Avg Energy (J)': round(avg_energies[i], 2),
                'Min Energy (J)': round(min_energies[i], 2)
            }
            csv_data.append(row)
        
        if csv_data:
            df = pd.DataFrame(csv_data)
            filename = f'baseline_simulation_data_run_{self.run_id}_{timestamp}.csv'
            df.to_csv(filename, index=False)
            print(f"[{self.run_id}] Exported baseline data for run {self.run_id} to CSV: {filename}")
            return filename
        return None

    def update_exploration_rate(self, drone_id, time_lap):
        tau = 1.0
        beta_m = self.learning_rate_initial + time_lap * self.step_size
        exploration_rate = np.exp(-beta_m / tau)
        
        self.drones[drone_id].exploration_rate = max(
            self.exploration_rate_min,
            min(self.exploration_rate_max, exploration_rate)
        )
        
        return self.drones[drone_id].exploration_rate

    def select_action_with_strategy(self, drone_id, current_action, trial_action, current_utility, trial_utility):
        max_utility = max(current_utility, trial_utility)
        normalized_current = current_utility - max_utility
        normalized_trial = trial_utility - max_utility
        
        exp_current = np.exp(-normalized_current / self.temperature)
        exp_trial = np.exp(-normalized_trial / self.temperature)
        
        total_exp = exp_current + exp_trial
        prob_current = exp_current / total_exp
        
        if random.random() < prob_current:
            selected_action = current_action
            selected_utility = current_utility
        else:
            selected_action = trial_action
            selected_utility = trial_utility
        
        return selected_action, selected_utility


def run_game_theoretic_experiment(num_runs=2, user_mobility_file="Dataset/Interpolation_user_movement_real_here.csv"):
    """Run multiple game-theoretic experiments - Each iteration creates its own CSV file"""
    print(f"Starting game-theoretic baseline experiment with {num_runs} runs...")
    print(f"User mobility file: {user_mobility_file}")
    print(f"Each iteration will create a separate CSV file\n")
    
    all_metrics = []
    all_csv_files = []
    
    for run in range(num_runs):
        print(f"\n{'='*80}")
        print(f"ITERATION {run+1}/{num_runs}")
        print(f"{'='*80}\n")
        
        env = simpy.Environment()
        network = DroneNetwork(env, run_id=run+1)
        
        metrics = network.run_game_theoretic_simulation(user_mobility_file)
        all_metrics.append(metrics)
        
        print(f"\n✓ Iteration {run+1} simulation complete")
    
    print(f"\n{'='*80}")
    print(f"BASELINE EXPERIMENT COMPLETED")
    print(f"{'='*80}")
    print(f"Total iterations: {num_runs}")
    print(f"CSV files created: {len(all_csv_files)}")
    for csv_file in all_csv_files:
        print(f"  - {csv_file}")
    print(f"{'='*80}\n")
    
    return all_metrics


if __name__ == "__main__":
    random.seed(123)
    np.random.seed(123)
    # load csv files
    #user_mobility_file = "Original_Dataset.csv"
    # Update this path to your actual CSV file
    user_mobility_file ="Synthetic_dataset_45.csv"
    all_metrics = run_game_theoretic_experiment(num_runs=50, user_mobility_file=user_mobility_file)
